# Traductor de txt a SQL

**Índice**   
1. [Imports](#imports)
2. [Cargamos los modelos](#cargamos-los-modelos)
3. [Creamos la base de datos](#creamos-la-base-de-datos)
4. [Mapeo de columnas](#mapeo-de-columnas)
5. [Sinonimos/palabras clave (ES/EN)](#sinonimos--palabras-clave-esen)
6. [Metricas](#metricas)
7. [Agrupaciones temporales](#agrupaciones-temporales)
8. [Funcion que detecta el idioma](#funcion-que-detecta-el-idioma) .
9. [Filtro de fecha](#filtro-de-fecha)
10. [Filtro minus y mayus](#filtro-minus-y-mayus)
11. [Filtro agrupaciones](#filtro-agrupaciones)
12. [Detección de where](#detección-de-where).
13. [Detección de group](#detección-de-group).
14. [Detección de filtro](#detección-de-filtro).
15. [Generador de SQL](#generador-de-SQL)
16. [Pruebas](#pruebas)

## Imports

In [134]:
import re
import spacy
from langdetect import detect
import pandas as pd

## Cargamos los modelos 

In [135]:
# Modelos
nlp_es = spacy.load("es_core_news_sm") # Español
nlp_en = spacy.load("en_core_web_sm") # English

## Creamos la base de datos

In [136]:
# Cargar los CSVs (ajusta las rutas a tus archivos locales)
clientes = pd.read_csv("./data/clientes_ecommerce.csv")
transacciones = pd.read_csv("./data/transacciones_ecommerce.csv")

In [137]:
df = pd.merge(transacciones, clientes, on="id_cliente", how="outer")
TABLE_NAME = "desafio_tripulaciones_db"
# IMPORTANTE: en tu df mergeado las columnas son las del CSV, aquí asumo que usas las españolas.

In [112]:
df

,id_transaccion,id_cliente,fecha_compra,producto,categoria_producto,precio_unitario,cantidad,importe_total,metodo_pago,coste_envio,coste_fabricacion,nombre,apellidos,email,pais,ciudad,edad,genero
0,1308.0,1,2024-03-21,Xiaomi 13,Móviles,1269.37,2.0,2538.74,bizum,6.64,723.00,Carlos,Sánchez Ramos,carlos.sanchez@gmail.com,España,Barcelona,29,M
1,1423.0,1,2023-09-04,Apple Watch Series 9,Relojes inteligentes,561.03,1.0,561.03,paypal,8.13,301.15,Carlos,Sánchez Ramos,carlos.sanchez@gmail.com,España,Barcelona,29,M
2,7682.0,1,2023-08-16,Auriculares Sony WH-1000XM5,Accesorios,224.99,1.0,224.99,tarjeta,5.90,67.77,Carlos,Sánchez Ramos,carlos.sanchez@gmail.com,España,Barcelona,29,M
3,8220.0,1,2024-10-26,Lenovo ThinkPad X1 Carbon,Portátiles,1562.94,2.0,3125.88,transferencia,24.97,1166.97,Carlos,Sánchez Ramos,carlos.sanchez@gmail.com,España,Barcelona,29,M
4,9001.0,1,2023-02-26,Huawei Watch GT 4,Relojes inteligentes,264.37,3.0,793.11,bizum,7.82,141.60,Carlos,Sánchez Ramos,carlos.sanchez@gmail.com,España,Barcelona,29,M
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15229,386.0,4999,2024-10-05,Apple Watch Series 9,Relojes inteligentes,437.23,2.0,874.46,transferencia,8.11,204.23,Alberto,Jiménez Ruiz,alberto.jimenez2@live.com,Reino Unido,Londres,49,F
15230,1958.0,4999,2024-09-04,Auriculares Sony WH-1000XM5,Accesorios,10.27,2.0,20.54,paypal,4.76,3.91,Alberto,Jiménez Ruiz,alberto.jimenez2@live.com,Reino Unido,Londres,49,F
15231,2311.0,4999,2024-02-21,Xiaomi 13,Móviles,739.49,2.0,1478.98,bizum,12.94,464.58,Alberto,Jiménez Ruiz,alberto.jimenez2@live.com,Reino Unido,Londres,49,F
15232,8452.0,4999,2023-09-20,HP Spectre x360,Portátiles,2215.25,1.0,2215.25,tarjeta,16.57,1340.05,Alberto,Jiménez Ruiz,alberto.jimenez2@live.com,Reino Unido,Londres,49,F


## Mapeo de columnas

In [113]:
COLS = {
    "id_cliente","nombre","apellidos","email","pais","ciudad","edad","genero",
    "id_transaccion","fecha_compra","producto","categoria_producto",
    "precio_unitario","cantidad","importe_total","metodo_pago",
    "coste_envio","coste_fabricacion"
}

## Sinonimos / palabras clave (ES/EN)

In [145]:
# Sinónimos / palabras clave -> columnas (ES/EN)
SYN_TO_COL = {
    "es": {
        # métricas
        "ventas": "importe_total",
        "ingresos": "importe_total",
        "beneficios": "importe_total",
        "facturacion": "importe_total",
        "importe": "importe_total",
        "total": "importe_total",
        "unidades": "cantidad",
        "cantidad": "cantidad",
        "precio": "precio_unitario",
        "envio": "coste_envio",
        "fabricacion": "coste_fabricacion",
        # dimensiones
        "pais": "pais",
        "ciudad": "ciudad",
        "producto": "producto",
        "categoria": "categoria_producto",
        "genero": "genero",
        "edad": "edad",
        "metodo": "metodo_pago",
        "pago": "metodo_pago",
        "fecha": "fecha_compra",
        "compra": "fecha_compra",
        "transaccion": "id_transaccion",
        "pedido": "id_transaccion",
        "cliente": "id_cliente",
    },
    "en": {
        # si el usuario pregunta en inglés, seguimos generando SQL con columnas ES
        # (porque tu dataset está en ES). Solo traducimos la intención.
        "sales": "importe_total",
        "revenue": "importe_total",
        "amount": "importe_total",
        "total": "importe_total",
        "units": "cantidad",
        "quantity": "cantidad",
        "price": "precio_unitario",
        "shipping": "coste_envio",
        "manufacturing": "coste_fabricacion",
        # dimensiones
        "country": "pais",
        "city": "ciudad",
        "product": "producto",
        "category": "categoria_producto",
        "gender": "genero",
        "age": "edad",
        "payment": "metodo_pago",
        "date": "fecha_compra",
        "purchase": "fecha_compra",
        "transaction": "id_transaccion",
        "order": "id_transaccion",
        "client": "id_cliente",
        "customer": "id_cliente",
    }
}

## Meses

In [147]:
MONTHS = {
    "es": {
        "enero": 1, "febrero": 2, "marzo": 3, "abril": 4,
        "mayo": 5, "junio": 6, "julio": 7, "agosto": 8,
        "septiembre": 9, "octubre": 10, "noviembre": 11, "diciembre": 12,
    },
    "en": {
        "january": 1, "february": 2, "march": 3, "april": 4,
        "may": 5, "june": 6, "july": 7, "august": 8,
        "september": 9, "october": 10, "november": 11, "december": 12,
    }
}

## Metricas

FALTAN ¿Varianza?

In [146]:
AGG_WORDS = {
    "es": {
        "avg": {"promedio", "media", "promediar"},
        "sum": {"suma", "total", "sumar", "sumatorio"},
        "count": {"cuantos", "cuantas", "numero", "numeros", "conteo", "contar"},
        "max": {"maximo", "maxima", "mayor", "pico", "tope"},
        "min": {"minimo", "minima", "menor", "bajo"},
        "median": {"mediana", "percentil", "percentil 50"},
        "mode": {"moda", "mas frecuente", "frecuente"},
        "std": {"desviacion", "desviacion estandar", "variacion"},
    },
    "en": {
        "avg": {"average", "avg", "mean"},
        "sum": {"sum", "total"},
        "count": {"count", "how", "many", "number"},
        "max": {"max", "maximum", "highest", "top"},
        "min": {"min", "minimum", "lowest"},
        "median": {"median", "percentile"},
        "mode": {"mode", "most frequent"},
        "std": {"std", "stddev", "standard deviation"}
    }
}

## Agrupaciones temporales

In [ ]:
TIME_GROUP_WORDS = {
    "es": {
        "quarter": {"trimestre", "trimestral", "trimestralmente", "cuatrimestre"},
        "month": {"mes", "mensual"},
        "year": {"año", "anual"},
    },
    "en": {
        "quarter": {"quarter", "qtr"},
        "month": {"month", "monthly"},
        "year": {"year", "yearly", "annual"},
    }
}

## Funcion que detecta el idioma

In [117]:
def detectar_idioma(texto: str):
    lang = detect(texto)
    if lang == "es":
        return nlp_es(texto), "es"
    elif lang == "en":
        return nlp_en(texto), "en"
    else:
        raise ValueError(f"Idioma no soportado: {lang}")

## Filtro de fecha

### Filtro año

In [ ]:
def _find_years(texto: str):
    return sorted(set(re.findall(r"\b(2023|2024)\b", texto)))

### SQL año

In [ ]:
def _year_range_condition_pg(years):
    # years es lista de strings [“2023”] o [“2023",“2024"]
    start_y = min(years)
    end_y = max(years)
    return (
        f"fecha_compra BETWEEN '{start_y}-01-01' AND '{end_y}-12-31'"
    )

### Rango meses

In [ ]:
def _find_month_ranges(texto: str, idioma: str):
    texto = texto.lower()
    pattern = (
        r"(enero|febrero|marzo|abril|mayo|junio|julio|agosto|septiembre|octubre|noviembre|diciembre|"
        r"january|february|march|april|may|june|july|august|september|october|november|december)"
        r"\s+(a|hasta|to)\s+"
        r"(enero|febrero|marzo|abril|mayo|junio|julio|agosto|septiembre|octubre|noviembre|diciembre|"
        r"january|february|march|april|may|june|july|august|september|october|november|december)"
        r"(?:\s+(\d{4}))?"
    )
    return re.findall(pattern, texto)

### SQL rango meses

In [ ]:
def _month_range_condition_pg(start_month, end_month, year):
    start_date = f"{year}-{start_month:02d}-01"
    # último día del mes final
    end_date = (
        f"date_trunc('month', DATE '{year}-{end_month:02d}-01') "
        f"+ INTERVAL '1 month - 1 day'"
    )
    return f"fecha_compra BETWEEN '{start_date}' AND {end_date}"

### Filtro un mes en concreto de un año en concreto

In [ ]:
def _find_single_month(texto: str, idioma: str):
    texto = texto.lower()
    for m, num in MONTHS[idioma].items():
        m_match = re.search(rf"\b{m}\b\s*(\d{{4}})?", texto)
        if m_match:
            year = m_match.group(1)
            return num, year
    return None

#### SQL un mes en concreto de un año en concreto

In [ ]:
def _single_month_condition_pg(month, year):
    start = f"{year}-{month:02d}-01"
    end = (
        f"date_trunc('month', DATE '{year}-{month:02d}-01') "
        f"+ INTERVAL '1 month - 1 day'"
    )
    return f"fecha_compra BETWEEN '{start}' AND {end}"

## Quitar tíldes

In [139]:
import unicodedata

def strip_accents(text: str) -> str:
    return ''.join(
        c for c in unicodedata.normalize('NFD', text)
        if unicodedata.category(c) != 'Mn'
    )

## Prepocesado minus y mayus

In [ ]:
def _normalize_tokens(doc):
    # lemmas en minúscula, sin puntuación/espacios
        return [
        strip_accents(t.lemma_.lower())
        for t in doc
        if not t.is_punct and not t.is_space
    ]

## Filtro ranking

In [ ]:
def detectar_ranking(tokens, idioma):
    top_words = {
        "es": {"top", "mejores", "mayores", "ranking"},
        "en": {"top", "best", "highest", "ranking"},
    }

    if any(t in tokens for t in top_words[idioma]):
        return True
    return False

## Detectar N top

In [ ]:
def detectar_limit(texto: str):
    m = re.search(r"\btop\s+(\d+)", texto.lower())
    if m:
        return int(m.group(1))
    return 5  # default razonable

## Filtro agrupaciones

In [121]:
def detectar_agregacion(tokens, idioma):
    # default: None (si no pide nada, se puede devolver *)
    for agg, words in AGG_WORDS[idioma].items():
        if any(w in tokens for w in words):
            return agg
    return None

## Detección de where

In [122]:
def detectar_metricas(tokens, idioma):
    # Busca la primera métrica "razonable"
    # Si menciona ventas/importe -> importe_total; unidades -> cantidad; etc.
    for tok in tokens:
        if tok in SYN_TO_COL[idioma]:
            col = SYN_TO_COL[idioma][tok]
            if col in {"importe_total", "cantidad", "precio_unitario", "coste_envio", "coste_fabricacion"}:
                return col
    # fallback: si habla de promedio sin métrica explícita, asumimos ventas
    return None

## Detección de group

In [138]:
def detectar_groupbys(tokens, idioma):
    group_cols = []
    # Tiempo
    if any(w in tokens for w in TIME_GROUP_WORDS[idioma]["quarter"]):
        group_cols.append("date_trunc('quarter', fecha_compra)")
    elif any(w in tokens for w in TIME_GROUP_WORDS[idioma]["month"]):
        group_cols.append("date_trunc('month', fecha_compra)")
    elif any(w in tokens for w in TIME_GROUP_WORDS[idioma]["year"]):
        group_cols.append("date_trunc('year', fecha_compra)")

    # Dimensiones típicas si el usuario las menciona
    dims_priority = ["pais", "ciudad", "categoria_producto", "producto", "genero", "metodo_pago"]
    mentioned = set()
    for tok in tokens:
        if tok in SYN_TO_COL[idioma]:
            mentioned.add(SYN_TO_COL[idioma][tok])

    for d in dims_priority:
        if d in mentioned and d in COLS:
            group_cols.append(d)
            
    # Elimina duplicados manteniendo orden
    seen = set()
    out = []
    for g in group_cols:
        if g not in seen:
            out.append(g)
            seen.add(g)
    return out

## Detección de filtro

In [ ]:
def detectar_filtros(doc, tokens, idioma):
    where = []
    text = doc.text.lower()

    # Rango de meses
    month_ranges = _find_month_ranges(text, idioma)
    if month_ranges:
        for m_start, _, m_end, year in month_ranges:
            y = year or _find_years(text)[0]
            where.append(
                _month_range_condition_pg(
                    MONTHS[idioma][m_start],
                    MONTHS[idioma][m_end],
                    y
                )
            )
        return where
    
    # Mes en concreto
    single_month = _find_single_month(text, idioma)
    if single_month:
        month, year = single_month
        if year:
            where.append(_single_month_condition_pg(month, year))
            return where

    # Año(s)
    years = _find_years(doc.text)
    if years:
        where.append(_year_range_condition_pg(years))
    
    # País / ciudad vía entidades LOC/GPE
    for ent in doc.ents:
        if ent.label_ in {"LOC", "GPE"}:
            # Heurística: si menciona "ciudad" cerca, filtra por ciudad; si no, por país.
            txt = ent.text.replace("'", "''")
            # Ventana simple alrededor de la entidad
            span_start = max(ent.start - 2, 0)
            span_end = min(ent.end + 2, len(doc))
            window = " ".join([t.lemma_.lower() for t in doc[span_start:span_end]])
            if ("ciudad" in window) or ("city" in window):
                where.append(f"ciudad = '{txt}'")
            else:
                where.append(f"pais = '{txt}'")
    # Producto / categoría por patrón "producto X" / "categoría Y"
    text_lower = doc.text.lower()
    # ES: "producto iphone", "categoría electronica"
    m_prod = re.search(r"(producto)\s+([a-z0-9_\-áéíóúñ ]{2,})", text_lower)
    if m_prod:
        val = m_prod.group(2).strip()
        # corta si aparecen conectores comunes
        val = re.split(r"\b(en|por|de|del|la|el|and|by|of)\b", val)[0].strip()
        where.append(f"producto LIKE ‘%{val.replace('\'','\'\'')}%")
    m_cat = re.search(r"(categor[ií]a)\s+([a-z0-9_\-áéíóúñ ]{2,})", text_lower)
    if m_cat:
        val = m_cat.group(2).strip()
        val = re.split(r"\b(en|por|de|del|la|el|and|by|of)\b", val)[0].strip()
        where.append(f"categoria_producto LIKE ‘%{val.replace('\'','\'\'')}%’")
    # Género simple (M/F, masculino/femenino, male/female)
    if re.search(r"\b(masculino|hombre|male|m)\b", text_lower):
        where.append("genero IN (‘M’,‘Masculino’,‘male’,‘Male’)")
    if re.search(r"\b(femenino|mujer|female|f)\b", text_lower):
        where.append("genero IN (‘F’,‘Femenino’,‘female’,‘Female’)")
    # Edad: "mayores de 30", "menores de 25"
    m_gt = re.search(r"(mayores de|más de|over|older than)\s+(\d{1,3})", text_lower)
    if m_gt:
        where.append(f"edad > {int(m_gt.group(2))}")
    m_lt = re.search(r"(menores de|menos de|under|younger than)\s+(\d{1,3})", text_lower)
    if m_lt:
        where.append(f"edad < {int(m_lt.group(2))}")
    # Dedup
    where_out = []
    seen = set()
    for w in where:
        if w not in seen:
            where_out.append(w)
            seen.add(w)
    return where_out

## Generador de SQL

COUNT(DISTINCT id_transaccion)

o subquery agregada antes del join

MODIFICAR COUNT, QUE NO SOLAMENTE SEA CON TRANSACCIONES

In [ ]:
def generar_sql(texto: str):
    doc, idioma = detectar_idioma(texto)
    tokens = _normalize_tokens(doc)
    agg = detectar_agregacion(tokens, idioma)
    metric = detectar_metricas(tokens, idioma)

    # Defaults "razonables"
    if agg in {"avg", "sum", "max", "min"} and metric is None:
        metric = "importe_total"

    if agg is None and metric is not None:
        # si menciona métrica pero no agg, asumimos SUM para ventas/unidades
        if metric in {"importe_total", "cantidad"}:
            agg = "sum"

    group_by = detectar_groupbys(tokens, idioma)
    where = detectar_filtros(doc, tokens, idioma)

    # SELECT
    select_parts = []
    if group_by:
        select_parts.extend(group_by)
    
    if agg == "avg":
        alias_metric = f"promedio_{metric}"
        select_parts.append(f"AVG({metric}) AS {alias_metric}")

    elif agg == "sum":
        alias_metric = f"total_{metric}"
        select_parts.append(f"SUM({metric}) AS {alias_metric}")

    elif agg == "max":
        alias_metric = f"max_{metric}"
        select_parts.append(f"MAX({metric}) AS {alias_metric}")

    elif agg == "min":
        alias_metric = f"min_{metric}"
        select_parts.append(f"MIN({metric}) AS min_{alias_metric}")
    
    elif agg == "median":
        alias_metric = f"mediana_{metric}"
        select_parts.append(f"PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY {metric}) AS {alias_metric}")
    
    elif agg == "mode":
        alias_metric = f"moda_{metric}"
        select_parts.append(f"MODE() WITHIN GROUP (ORDER BY {metric}) AS {alias_metric}")
    
    elif agg == "std":
        alias_metric = f"std_{metric}"
        select_parts.append(f"STDDEV_POP({metric}) AS {alias_metric}")

    elif agg == "count": # MODIFICAR, HACERLO DINÁMICO
        # Si pide conteo, contamos transacciones por defecto
        alias_metric = "conteo_transacciones"
        select_parts.append(f"COUNT(id_transaccion) AS {alias_metric}")

    else:
        # Sin intención: devuelve columnas principales (evita SELECT *)
        select_parts.append("id_transaccion")
        select_parts.append("fecha_compra")
        select_parts.append("importe_total")
        select_parts.append("cantidad")

    sql = "SELECT " + ", ".join(select_parts) + f" FROM {TABLE_NAME}"

    if where:
        sql += " WHERE " + " AND ".join(where)

    if group_by:
        sql += " GROUP BY " + ", ".join(group_by)

    if detectar_ranking(tokens, idioma) and alias_metric:
        sql += f" ORDER BY {alias_metric} DESC"
        sql += f" LIMIT {detectar_limit(texto)}"
    sql += ";"
    return sql

## Pruebas

En la mitad de la query que hada un outer join entre mi tabla cleinte y transacciones

SELECT *

FROM transacciones

OUTTER JOIN cliente ON transacciones.id_cliente = cliente.id_cliente;

In [126]:
print(generar_sql("¿Cuántas transacciones hubo en 2024?"))

SELECT COUNT(id_transaccion) AS conteo_transacciones FROM desafio_tripulaciones_db WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31';


In [127]:
print(generar_sql("¿Cuántas transacciones hubo en España en el año 2023?"))

SELECT date_trunc('year', fecha_compra), COUNT(id_transaccion) AS conteo_transacciones FROM desafio_tripulaciones_db WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31' AND pais = ‘España‘ GROUP BY date_trunc('year', fecha_compra);


In [128]:
print(generar_sql("Ventas máximas por país en enero"))

SELECT id_transaccion, fecha_compra, importe_total, cantidad FROM desafio_tripulaciones_db;


In [129]:
print(generar_sql("Cuántas transacciones en España en 2023 por mes"))

SELECT date_trunc('month', fecha_compra), COUNT(id_transaccion) AS conteo_transacciones FROM desafio_tripulaciones_db WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31' AND pais = ‘España‘ GROUP BY date_trunc('month', fecha_compra);


In [130]:
print(generar_sql("Total sales in 2024 by country"))

SELECT pais, SUM(importe_total) AS total_importe_total FROM desafio_tripulaciones_db WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' GROUP BY pais;


In [131]:
print(generar_sql("Muéstrame el promedio de ventas en 2023 por trimestre"))

SELECT date_trunc('quarte', fecha_compra), AVG(importe_total) AS promedio_importe_total FROM desafio_tripulaciones_db WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31' GROUP BY date_trunc('quarte', fecha_compra);


In [132]:
print(generar_sql("Número de transacciones en México en 2024"))

SELECT COUNT(id_transaccion) AS conteo_transacciones FROM desafio_tripulaciones_db WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' AND pais = ‘México‘;


In [133]:
print(generar_sql("¿Cuántas transacciones hubo en España en el año 2023?"))

SELECT date_trunc('year', fecha_compra), COUNT(id_transaccion) AS conteo_transacciones FROM desafio_tripulaciones_db WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31' AND pais = ‘España‘ GROUP BY date_trunc('year', fecha_compra);
